# README
- **Author**: RUEI-HSIANG, CHANG（張睿庠；B12303030；AI協助：Codex）
- **Created At**: 2026-11-14
- **Last Modified At**: 2026-11-14

## What does this file do?
合併280位候選人（臺北議員98、高雄議員101、全國縣市長81）的既有社群帳號資料，補入新來源連結。平台Facebook、Instagram、Threads，共840列工作單。既有快照不改寫，新版完整表為 `candidate_social_links_scope_20261114.csv`。按candidate_key連結，不以姓名跨候選人合併。

本輪299次原始來源請求（其中199位缺漏候選人百科嘗試）、另55次官方頁面及widget擷取；包含失敗。Wiki同名人物以縣市與政治背景篩選，僅收候選人具名的帳號外連；保留未覆核標記。排除分享按鈕、搜尋頁、組織共用頁尾、國民黨mykmt及同名藝人吳宗憲。高雄官方候選人頁iframe href支持來源歸屬，但未開啟目標帳號驗證。不得將來源抽取視為人工讀完全文或確認目前可用。目錄／百科仍只是lead，複數帳號保留供覆核，不猜Threads名稱。

## What does this file take?
七份既有CSV與raw manifest，精確路徑／hash見PINS。新觀察記在raw link_observations.json，每筆含來源路徑與hash；來源只保存擷取文字／連結，不含cookie、登入資料或script配置。來源文字未逐篇人工完整覆核，不能當研究編碼事實。

## What does this file output?
- `data/processed/candidate_social_links_scope_20261114.csv`：280列寬表，三平台urls與狀態。
- `data/processed/social_link_worklist_scope_20261114.csv`：840列，前後缺漏、新增旗標、來源與核對層級。
- `data/processed/social_link_evidence_added_20261114.csv`：61筆新來源觀察（55個候選人×平台缺漏填入）。
- `data/processed/social_link_missing_summary_20261114.csv`：12列各群×平台缺漏統計。
- Role: collection worklist / evidence / summary；Documentation: `data/processed/social_links_scope_20261114_documentation.ipynb`。
- Verification: 執行輸出記錄保存檔hash；逐值重讀、鍵、列數及未減少既有涵蓋檢查。程式重建四個本輪專用輸出。
- 未取得連結保持缺漏，不當成無帳號；未擷取貼文、不判斷活躍、不重跑分群。


### 2026-11-14 第二輪續查
另查官方臺北議會名冊、32個現任議員頁與52個個人網站、城市選舉目錄、政治人物消歧義頁及kiang/elections歷史Facebook目錄。本輪新增14筆來源觀察，填入12個Facebook缺漏，全部仍待覆核（11位只具歷史來源，1位另有百科連結；尚未確認現今有效）。歷史目錄按姓名與縣市連結，原職務可能不同，確切年份待查；保留原URL並標示historical_unverified_lead。未找到新IG／Threads，不猜帳號。累計61筆新增來源觀察，填入55個候選人平台缺漏。新raw manifest與每個來源hash亦固定核對。


In [1]:
from pathlib import Path
import csv,json,hashlib,re,collections
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists())
PINS={'data/processed/candidate_master_current.csv': 'f4806b229118b4f67383e4ed081144bffc0b81404600ffce646df6017c23eea4', 'data/processed/candidate_fields_current.csv': '74ef1f3a77d24f73a5f3682682b18c97ca55843e7879ada7c1cc86b57dd0730c', 'data/processed/candidate_social_account_evidence_current.csv': '7b53408d0db28ec76e50571f94307aa71591d299a6cd8dab8c03e6f3dce07666', 'data/processed/s1_tk_account_evidence_20261110.csv': 'e9728684b2265049ee7e316ea054c65511ada58926ad2102cac70e469ece74ba', 'data/processed/strategy_account_evidence_20261111.csv': 'f4722993264840062e590bb176400e17b7121fc65c5a3cdfd4b193c56eeddd95', 'data/processed/s1_tk_account_worklist_20261110.csv': '5124f363555ee4bd7f4b3ba91612cb6790b8c40f02b9df78e452452f87a06ad7', 'data/processed/strategy_account_worklist_20261111.csv': '2dac8a97059cc6588270263b0832890d63c3b2ffcb077aeea2f4484657df0fd1', 'data/raw/social_links_20261114/file_manifest.json': 'afb4462abb3291d626d809097b8b5b6b584f4810082437a0bbbd742ea9cbc7b3'}
PINS['data/raw/social_links_20261114_round2/file_manifest.json']='18573434982326289755a359e7afa495cc60b9edc0659e9502a10e6af78b1223'
for p,h in PINS.items():assert hashlib.sha256((ROOT/p).read_bytes()).hexdigest()==h,p
manifest=json.loads((ROOT/'data/raw/social_links_20261114/file_manifest.json').read_text())
for m in manifest:assert hashlib.sha256((ROOT/m['path']).read_bytes()).hexdigest()==m['sha256']
extra_manifest=json.loads((ROOT/'data/raw/social_links_20261114_round2/file_manifest.json').read_text())
for m in extra_manifest:assert hashlib.sha256((ROOT/m['path']).read_bytes()).hexdigest()==m['sha256']
def read(n):
 with (ROOT/'data/processed'/n).open(encoding='utf-8-sig',newline='') as f:return list(csv.DictReader(f))
master=read('candidate_master_current.csv')
target=[r for r in master if '長' in r['office_type'] or(r['county_city'] in ['臺北市','高雄市'] and '議員' in r['office_type'])]
assert len(target)==280 and len({r['candidate_key'] for r in target})==280
keys={r['candidate_key'] for r in target};platforms=['Facebook','Instagram','Threads']
urls=collections.defaultdict(set);verified=collections.defaultdict(set);supported=collections.defaultdict(set);sources=collections.defaultdict(set)
for r in read('candidate_fields_current.csv'):
 if r['candidate_key'] in keys and r['field'] in platforms:
  k=(r['candidate_key'],r['field']);urls[k].update(re.findall(r'https?://[^\s|]+',r['existing_value']))
for name in ['candidate_social_account_evidence_current.csv','s1_tk_account_evidence_20261110.csv','strategy_account_evidence_20261111.csv']:
 for r in read(name):
  if r['candidate_key'] in keys and r['platform'] in platforms and r['account_url'].startswith('http'):
   k=(r['candidate_key'],r['platform']);urls[k].add(r['account_url']);sources[k].add(r['source_url'])
   if r.get('attribution_status')=='source_attribution_verified':verified[k].add(r['account_url'])
for name,col in [('s1_tk_account_worklist_20261110.csv','account_urls'),('strategy_account_worklist_20261111.csv','lead_account_urls')]:
 for r in read(name):
  if r['candidate_key'] in keys:
   k=(r['candidate_key'],r['platform']);urls[k].update(json.loads(r[col]));verified[k].update(json.loads(r['verified_account_urls']))
before={k:bool(urls[k]) for k in [(r['candidate_key'],p) for r in target for p in platforms]}
observations=json.loads((ROOT/'data/raw/social_links_20261114/link_observations.json').read_text())
observations+=json.loads((ROOT/'data/raw/social_links_20261114_round2/link_observations.json').read_text())
observation_columns=sorted(set().union(*(r.keys() for r in observations)))
observations=[{c:r.get(c,'') for c in observation_columns} for r in observations]
for r in observations:
 assert r['candidate_key'] in keys and not before[(r['candidate_key'],r['platform'])]
 k=(r['candidate_key'],r['platform']);urls[k].add(r['account_url']);sources[k].add(r['source_url'])
 if r['status']=='institutional_source_link_target_unchecked':supported[k].add(r['account_url'])
work=[];wide=[]
for r in target:
 w={k:r[k] for k in ['candidate_key','candidate_name','county_city','office_type']}
 for p in platforms:
  k=(r['candidate_key'],p);u=sorted(urls[k]);v=sorted(verified[k]);sp=sorted(supported[k]);ss=sorted(sources[k])
  status='source_attribution_verified_previous_round' if v else ('institutional_source_link_target_unchecked' if sp else ('unverified_lead' if u else 'not_located'))
  w[p+'_urls']=json.dumps(u,ensure_ascii=False);w[p+'_status']=status
  work.append(dict(candidate_key=r['candidate_key'],candidate_name=r['candidate_name'],county_city=r['county_city'],office_type=r['office_type'],platform=p,account_urls=json.dumps(u,ensure_ascii=False),verified_account_urls=json.dumps(v,ensure_ascii=False),institutional_source_account_urls=json.dumps(sp,ensure_ascii=False),source_urls=json.dumps(ss,ensure_ascii=False),link_status=status,missing_before=int(not before[k]),missing_after=int(not u),newly_filled=int(not before[k] and bool(u)),multiple_urls_review_required=int(len(u)>1),checked_date='2026-11-14',target_account_access='not_checked_this_round'))
 wide.append(w)
summary=[]
for group,people in [('Taipei councilors',[r for r in target if r['county_city']=='臺北市' and '議員' in r['office_type']]),('Kaohsiung councilors',[r for r in target if r['county_city']=='高雄市' and '議員' in r['office_type']]),('All mayoral candidates',[r for r in target if '長' in r['office_type']]),('TOTAL',target)]:
 kk={r['candidate_key'] for r in people}
 for p in platforms:
  ws=[r for r in work if r['candidate_key'] in kk and r['platform']==p]
  summary.append(dict(group=group,platform=p,candidates=len(people),missing_before=sum(r['missing_before'] for r in ws),newly_filled=sum(r['newly_filled'] for r in ws),missing_after=sum(r['missing_after'] for r in ws)))
outputs={'candidate_social_links_scope_20261114.csv':wide,'social_link_worklist_scope_20261114.csv':work,'social_link_evidence_added_20261114.csv':observations,'social_link_missing_summary_20261114.csv':summary}
assert len(work)==840 and len({(r['candidate_key'],r['platform']) for r in work})==840
assert sum(r['newly_filled'] for r in work)==55
assert all(r['missing_after']<=r['missing_before'] for r in work)
checks=[]
for name,rows in outputs.items():
 path=ROOT/'data/processed'/name
 with path.open('w',encoding='utf-8-sig',newline='') as f:
  w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
 back=read(name);assert len(back)==len(rows) and all(back[i]=={k:str(v) for k,v in r.items()} for i,r in enumerate(rows))
 checks.append(dict(path=str(path.relative_to(ROOT)),rows=len(back),sha256=hashlib.sha256(path.read_bytes()).hexdigest()))
print(json.dumps(checks,ensure_ascii=False,indent=2))
for r in summary:print(r)
print('New links:',len(observations),'newly filled candidate-platform pairs:',sum(r['newly_filled'] for r in work))


[
  {
    "path": "data/processed/candidate_social_links_scope_20261114.csv",
    "rows": 280,
    "sha256": "ac9dab6e14504395106de57eb9d2861dc029e400a743448ec42e756098d68e89"
  },
  {
    "path": "data/processed/social_link_worklist_scope_20261114.csv",
    "rows": 840,
    "sha256": "0e70683029b03213783fecda1c8f77baa64887a758797711561abf2603f83b7f"
  },
  {
    "path": "data/processed/social_link_evidence_added_20261114.csv",
    "rows": 61,
    "sha256": "79e97555c46b0ee33f5670119d645ecdb36656ae690ad26b3970d05fcfa9b4ac"
  },
  {
    "path": "data/processed/social_link_missing_summary_20261114.csv",
    "rows": 12,
    "sha256": "d143ff8f961353a2353e28a5ceb1e99bd409dd70ce05ef35b2ae1831f9bc05b6"
  }
]
{'group': 'Taipei councilors', 'platform': 'Facebook', 'candidates': 98, 'missing_before': 36, 'newly_filled': 10, 'missing_after': 26}
{'group': 'Taipei councilors', 'platform': 'Instagram', 'candidates': 98, 'missing_before': 51, 'newly_filled': 3, 'missing_after': 48}
{'group': 'Taipe